In [0]:
dbutils.library.restartPython()

In [ ]:

import json

# Create widget to receive table_config from For Each
dbutils.widgets.text("table_config", "{}")
table_config_json = dbutils.widgets.get("table_config")

print(f"Raw parameter: {table_config_json[:200]}...")  # Show first 200 chars

config = json.loads(table_config_json)
print(f"\n Processing table: {config.get('source_table', 'UNKNOWN')}")

In [ ]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig
)
from databricks.labs.lakebridge.reconcile.recon_config import Filters, Table, Transformation
from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException
from databricks.sdk import WorkspaceClient
from databricks.labs.lakebridge import __version__
from dataclasses import dataclass

@dataclass
class TableRecon:
    source_schema: str
    target_catalog: str
    target_schema: str
    tables: list[Table]
    source_catalog: str | None = None

ws = WorkspaceClient(product="lakebridge", product_version=__version__)
print(" Libraries imported and workspace client initialized")

In [ ]:
try:
    print(f"Processing table: {config['source_table']}")
    
    reconcile_config = ReconcileConfig(
        data_source=config["data_source"],
        report_type=config["report_type"].lower(),
        secret_scope=config["secret_scope"],
        database_config=DatabaseConfig(
            source_catalog=config["source_catalog"],
            source_schema=config["source_schema"],
            target_catalog=config["target_catalog"],
            target_schema=config["target_schema"]
        ),
        metadata_config=ReconcileMetadataConfig(
            catalog=config["target_catalog"],
            schema="lakebridge_recon"
        ))

    # Handle filters
    filters_column = config.get("filters_column")
    source_condition = config.get("source_filters_condition")
    target_condition = config.get("target_filters_condition")
    
    table_filters = None
    if filters_column and source_condition and target_condition:
        if str(filters_column).strip() and str(source_condition).strip():
            table_filters = Filters(
                source=f"lower({filters_column}) {source_condition}",
                target=f"lower({filters_column}) {target_condition}"
            )
            print(f" Filters: {filters_column}")

    table_recon = TableRecon(
        source_schema=config["source_schema"],
        target_catalog=config["target_catalog"],
        target_schema=config["target_schema"],
        tables=[
            Table(
                source_name=config["source_table"],
                target_name=config["target_table"],
                join_columns=config["join_columns"],
                filters=table_filters
            )
        ]
    )

    print(f" Starting recon for {config['source_table']}...")
    result = TriggerReconService.trigger_recon(
        ws=ws, 
        spark=spark, 
        table_recon=table_recon, 
        reconcile_config=reconcile_config
    )
    
    print(f" SUCCESS | Table: {config['source_table']} | Recon ID: {result.recon_id}")
    
except ReconciliationException as e:
    print(f"  RECONCILIATION COMPLETED WITH ISSUES")
    print(f"Table: {config['source_table']}")
    
    recon_output = e.reconcile_output if hasattr(e, 'reconcile_output') else (e.args[1] if len(e.args) > 1 else None)
    
    if recon_output:
        print(f"Recon ID: {recon_output.recon_id}")
        
        for table_result in recon_output.results:
            print(f"\n Reconciliation Results:")
            print(f"  Row Match: {' PASS' if table_result.status.row else ' FAIL'}")
            print(f"  Column Match: {' PASS' if table_result.status.column else ' FAIL'}")
            print(f"  Schema Match: {' PASS' if table_result.status.schema else ' FAIL'}")
        
        print(f"\n  Continuing with remaining tables...")
    
except Exception as e:
    print(f" FAILED | Table: {config.get('source_table', 'unknown')} | Error: {e}")
    raise